Cell 0: Imports

In [2]:
# Import all required libraries

import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import os

Cell 1: Tickers

In [3]:
# Part 1: Extract S&P 500 tickers from Wikipedia using Pandas
# User-Agent is required to avoid HTTP 403 blocking from Wikipedia

retrieval_date = '2026-11-21'
if os.path.exists(f'data/sp500_{retrieval_date}.csv'):
  sp500_df = pd.read_csv(f'data/sp500_{retrieval_date}.csv')
else:
  url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
  table = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})
  sp500_df = table[0]

  sp500_df = sp500_df[['Symbol', 'Security', 'GICS Sector', 'Date added']].copy()
  os.makedirs('data', exist_ok=True)

  sp500_df['Symbol'] = sp500_df['Symbol'].str.replace('.', '-', regex=False)
  sp500_df.to_csv(f'data/sp500_{retrieval_date}.csv', index=False)

tickers = sp500_df['Symbol'].tolist()
print('Retrieval date:',retrieval_date)
print('Total tickers extracted:', len(tickers))
sp500_df.head(10)

Retrieval date: 2026-11-21
Total tickers extracted: 503


,Symbol,Security,GICS Sector,Date added
0,MMM,3M,Industrials,1957-03-04
1,AOS,A. O. Smith,Industrials,2017-07-26
2,ABT,Abbott Laboratories,Health Care,1957-03-04
3,ABBV,AbbVie,Health Care,2012-12-31
4,ACN,Accenture,Information Technology,2011-07-06
5,ADBE,Adobe Inc.,Information Technology,1997-05-05
6,AMD,Advanced Micro Devices,Information Technology,2017-03-20
7,AES,AES Corporation,Utilities,1998-10-02
8,AFL,Aflac,Financials,1999-05-28
9,A,Agilent Technologies,Health Care,2000-06-05


In [4]:
# Part2: Load Daily Market Data via yfinance
# Download 'Close','Adj Close','Volume', 'Dividends', 'Stock Splits' and ^GSPC benchmark for all S&P 500 tickers
# Period: 01-01-2021 to 01-01-2025
start, end = '2021-01-01', '2025-01-01'
prices_file = f'data/sp500_prices_{retrieval_date}.csv.gz'

if os.path.exists(prices_file):
  df = pd.read_csv(prices_file, header=[0,1], index_col=0, parse_dates=True)
else:
  df = yf.download(tickers + ['^GSPC'], start=start, end=end, auto_adjust=False, actions=True)
  df = df[['Close','Adj Close','Volume', 'Dividends', 'Stock Splits']]
  df.to_csv(prices_file)

failed = df['Adj Close'].columns[df['Adj Close'].isna().all()].tolist()
print('pandas', pd.__version__, '\nyfinance', yf.__version__)
print('Failed tickers:', failed, 'Failed count:', len(failed))
print('Shape:', df.shape)
df.head()

[*********************100%***********************]  504 of 504 completed
ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['SNDK', 'VYLR', 'HONA', 'Q', 'FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


pandas 2.2.3 
yfinance 0.2.66
Failed tickers: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR'] Failed count: 5
Shape: (1005, 2510)


Price            Close                                                  \
Ticker               A        AAPL        ABBV        ABNB         ABT   
Date                                                                     
2021-01-04  118.639999  129.410004  105.410004  139.149994  109.110001   
2021-01-05  119.610001  131.009995  106.500000  148.300003  110.459999   
2021-01-06  122.889999  126.599998  105.580002  142.770004  110.230003   
2021-01-07  126.160004  130.919998  106.709999  151.270004  111.300003   
2021-01-08  127.059998  132.050003  107.269997  149.770004  111.610001   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2021-01-04  34.900002  256.459991  485.339996  146.020004  49.869999  ...   
2021-01-05  35.040001  257.920013  485.690002  148.630005  50.840000  ...   
2021-01-06  36.580002  260.739990  466.309998  149.300003  52.900002  ...   
2021-01-07  36.240002  263.200012  477.739990  155.610001  52.430000  ...   
2021-01-08  36.439999  264.160004  485.100006  156.740005  52.570000  ...   

Price      Stock Splits                                                
Ticker             WYNN  XEL  XOM  XYL  XYZ  YUM  ZBH ZBRA  ZTS ^GSPC  
Date                                                                   
2021-01-04          0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0  
2021-01-05          0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0  
2021-01-06          0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0  
2021-01-07          0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0  
2021-01-08          0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0  

[5 rows x 2510 columns]

In [5]:
# Part 3
excluded = {ticker:'no price data in study period' for ticker in failed}
print('Excluded tickers:', excluded)
print('Duplicate days:', df.index.duplicated().sum())

missing = df['Adj Close'].drop(columns=excluded).isna().sum()
print(missing[missing>0])



Excluded tickers: {'FDXF': 'no price data in study period', 'HONA': 'no price data in study period', 'Q': 'no price data in study period', 'SNDK': 'no price data in study period', 'VYLR': 'no price data in study period'}
Duplicate days: 0
Ticker
APP      70
CEG     263
COIN     69
EXE      26
GEHC    492
GEV     812
HOOD    143
RDDT    808
SOLV    811
VLTO    692
dtype: int64


In [6]:
late = missing[missing > 0]
leading = (df['Adj Close'][late.index].notna().cumsum() ==0).sum()
print((leading == late).all())

True


In [7]:
print(df['Volume'].index[df['Volume']['FERG']<=0])
print(df['Volume'].index[df['Volume']['SW']<=0])

DatetimeIndex(['2021-01-07', '2021-01-08', '2021-01-12', '2021-01-13',
               '2021-01-14', '2021-01-15', '2021-01-20', '2021-01-21',
               '2021-01-22', '2021-01-25', '2021-01-26', '2021-01-28',
               '2021-01-29', '2021-02-01', '2021-02-02', '2021-02-03',
               '2021-02-09', '2021-02-10', '2021-02-11', '2021-02-12',
               '2021-02-16', '2021-02-19', '2021-02-22', '2021-02-23',
               '2021-03-01', '2021-03-04', '2021-03-05'],
              dtype='datetime64[ns]', name='Date', freq=None)
DatetimeIndex(['2021-01-05', '2021-01-06', '2021-01-12', '2021-01-13',
               '2021-01-19', '2021-01-20', '2021-01-21', '2021-01-22',
               '2021-01-25', '2021-01-26',
               ...
               '2024-06-05', '2024-06-07', '2024-06-10', '2024-06-11',
               '2024-06-12', '2024-06-13', '2024-06-17', '2024-06-25',
               '2024-07-02', '2024-07-05'],
              dtype='datetime64[ns]', name='Date', length=424, f

In [8]:
excluded.update({ticker: 'listed during study period' for ticker in late.index})
print(len(excluded))

15


In [9]:
excluded.update({'FERG': 'listed during study period (zero volume before listing)',
                 'SW': 'listed during study period (zero volume before listing)'})
print(len(excluded))

17


In [10]:
valid = df.drop(columns=excluded, level=1)
print('Valid left:', valid['Adj Close'].shape[1])
print('Missing:', valid.isna().sum().sum())
print('Non-positive values:', (valid[['Close', 'Adj Close']] <=0).sum().sum())

bad_volume = (valid['Volume'] <= 0).sum()
print('Bad volume:', bad_volume[bad_volume > 0])

Valid left: 487
Missing: 0
Non-positive values: 0
Bad volume: Ticker
^GSPC    1
dtype: int64


In [11]:
split_tickers = valid['Stock Splits'].columns[(valid['Stock Splits']>0).any()].tolist()
print('Tickers split:', len(split_tickers))

close_returns = valid['Close'].pct_change(fill_method=None)
split_day_move = close_returns[valid['Stock Splits'] > 0].abs().max().max()
print('Largest Close move on a split date:', round(split_day_move,4))

dividend_payers = (valid['Dividends'] > 0).any().sum()
print('Shares paying dividends:', dividend_payers)

jumps = close_returns.abs().max()
big_jumps = jumps[jumps>0.4]
print('One-day moves above 40%',big_jumps.round(3).to_dict())

Tickers split: 53
Largest Close move on a split date: 0.0766
Shares paying dividends: 392
One-day moves above 40% {'BE': 0.592, 'CVNA': 0.56, 'DXCM': 0.407, 'GL': 0.531}


In [12]:
print('Big movers that also split: ', set(big_jumps.index) & set(split_tickers))
print('Dates of big moves: ', valid['Close'][big_jumps.index].pct_change(fill_method=None).abs().idxmax())

Big movers that also split:  {'DXCM'}
Dates of big moves:  Ticker
BE     2024-11-15
CVNA   2023-06-08
DXCM   2024-07-26
GL     2024-04-11
dtype: datetime64[ns]


In [13]:
dxcm = valid['Stock Splits']['DXCM']
print('DXCM split:', dxcm[dxcm>0])

DXCM split: Date
2022-06-13    4.0
Name: DXCM, dtype: float64


In [14]:
exclusions = pd.DataFrame(excluded.items(), columns=['Symbol', 'Reason'])
exclusions = exclusions.merge(sp500_df, on='Symbol', how='left')
exclusions.to_csv(f'data/sp500_exclusions_{retrieval_date}.csv', index=False)
exclusions

,Symbol,Reason,Security,GICS Sector,Date added
0,FDXF,no price data in study period,FedEx Freight,Industrials,2026-06-01
1,HONA,no price data in study period,Honeywell Aerospace,Industrials,2026-06-29
2,Q,no price data in study period,Qnity Electronics,Information Technology,2025-11-03
3,SNDK,no price data in study period,Sandisk,Information Technology,2025-11-28
4,VYLR,no price data in study period,Vylor,Consumer Staples,2026-10-01
5,APP,listed during study period,AppLovin,Communication Services,2025-09-22
6,CEG,listed during study period,Constellation Energy,Utilities,2022-02-02
7,COIN,listed during study period,Coinbase,Financials,2025-05-19
8,EXE,listed during study period,Expand Energy,Energy,2025-03-24
9,GEHC,listed during study period,GE HealthCare,Health Care,2023-01-04


In [15]:
# Part 4
weekly_prices = valid['Adj Close'].resample('W-FRI').last()
weekly_prices = weekly_prices.loc[:'2024-12-27'] # 2024-12-27 is the last complete week
weekly_returns = weekly_prices.pct_change(fill_method=None).iloc[1:]

mean_return = weekly_returns.drop(columns='^GSPC').mean()
print('NaN count:', weekly_returns.isna().sum().sum())
print('Number of weeks:', len(weekly_returns))

NaN count: 0
Number of weeks: 207


In [16]:
# Part 5
market = weekly_returns['^GSPC']
beta = weekly_returns.cov()['^GSPC'] / market.var()
print('Market beta:', beta['^GSPC'])

beta = beta.drop('^GSPC')

print('Paired weeks per share',weekly_returns.notna().sum().unique())

Market beta: 1.0
Paired weeks per share [207]


In [17]:
# Part 6
volume = valid['Volume'].drop(columns='^GSPC')
close = valid['Close'].drop(columns='^GSPC')

obs_date = pd.Timestamp('2024-12-31')
print('Common end date:',volume.index[-1] == obs_date)

avg_volume = volume.shift(1).rolling(20).mean()
high = volume > 2 * avg_volume
print('Total high-voulme days:',high.sum().sum())

Common end date: True
Total high-voulme days: 18280


In [18]:
frequency = high.tail(60).sum()
monetary = (close*volume).tail(60).sum()
last_event = high.iloc[::-1].idxmax().where(high.any())
recency = (obs_date - last_event).dt.days
rfm = pd.DataFrame({'Recency': recency,'Frequency': frequency, 'Monetary': monetary})

print('Missing recency:',rfm.index[rfm['Recency'].isna()])
rfm.describe()

Missing recency: Index([], dtype='object', name='Ticker')


,Recency,Frequency,Monetary
count,486.000000,486.000000,4.860000e+02
mean,20.407407,2.989712,3.940419e+10
std,21.342438,1.330800,1.291388e+11
min,4.000000,0.000000,1.297483e+09
25%,11.000000,2.000000,1.019517e+10
50%,11.000000,3.000000,1.668382e+10
75%,14.750000,4.000000,3.227689e+10
max,222.000000,8.000000,1.847697e+12


In [19]:
print('Missing baselines last 60 sessions',avg_volume.tail(60).isna().sum().sum())
features = pd.DataFrame({'Recency': recency,'Frequency': frequency, 'Monetary': monetary, 'Beta': beta, 'Mean_return': mean_return})

features.to_csv(f'data/features_{retrieval_date}.csv')
print('Features skew:',features.skew())
features.describe()

Missing baselines last 60 sessions 0
Features skew: Recency         3.645784
Frequency       0.424735
Monetary       11.683797
Beta            1.103649
Mean_return     0.867516
dtype: float64


,Recency,Frequency,Monetary,Beta,Mean_return
count,486.000000,486.000000,4.860000e+02,486.000000,486.000000
mean,20.407407,2.989712,3.940419e+10,0.973695,0.002857
std,21.342438,1.330800,1.291388e+11,0.407339,0.002652
min,4.000000,0.000000,1.297483e+09,0.055453,-0.004468
25%,11.000000,2.000000,1.019517e+10,0.666599,0.001182
50%,11.000000,3.000000,1.668382e+10,0.950609,0.002619
75%,14.750000,4.000000,3.227689e+10,1.179538,0.004207
max,222.000000,8.000000,1.847697e+12,3.778560,0.017772


In [20]:
day = last_event['AAPL']

print('AAPL last event:', day.date())
print('Observation date', obs_date.date())
print('Recency (days):', recency['AAPL'])

print('Volume on event day:', volume.loc[day, 'AAPL'])
print('20-session average:', avg_volume.loc[day, 'AAPL'])
print('Threshold (2x average):', 2*avg_volume.loc[day, 'AAPL'])

manual_avg = volume['AAPL'].loc[:day].iloc[-21:-1].mean()
print('Manual 20-session average:', manual_avg)

print('Frequency:', frequency['AAPL'])
print('Monetary (USD):', monetary['AAPL'])

AAPL last event: 2024-12-20
Observation date 2024-12-31
Recency (days): 11
Volume on event day: 147495300
20-session average: 45004605.0
Threshold (2x average): 90009210.0
Manual 20-session average: 45004605.0
Frequency: 2
Monetary (USD): 623303377706.5842


In [21]:
print('Shares added after start:', (pd.to_datetime(sp500_df.set_index('Symbol').loc[features.index, 'Date added']) > '2021-01-01').sum())

Shares added after start: 72


In [22]:
print('Shares whose last event was 20 Dec 2024:', (last_event == '2024-12-20').sum())

Shares whose last event was 20 Dec 2024: 353


In [23]:
print('Shares with F = 0:', (frequency == 0).sum())

Shares with F = 0: 10


In [24]:
print('^GSPC mean weekly return:', weekly_returns['^GSPC'].mean())

^GSPC mean weekly return: 0.002398425852406388


In [25]:
x = weekly_returns['^GSPC']
y = weekly_returns['AAPL']

print('np.polyfit beta (AAPL):', np.polyfit(x, y, 1)[0])
print('cov/var beta (AAPL):', beta['AAPL'])

np.polyfit beta (AAPL): 1.1748360046001136
cov/var beta (AAPL): 1.1748360046001136


In [26]:
p0 = weekly_prices['AAPL'].iloc[0]
p1 = weekly_prices['AAPL'].iloc[1]

print('Dates:', weekly_prices.index[0].date(), '->', weekly_prices.index[1].date())
print('Manual return:', p1 / p0 - 1)
print('pct_change return:', weekly_returns['AAPL'].iloc[0])

Dates: 2021-01-08 -> 2021-01-15
Manual return: -0.03718309110892759
pct_change return: -0.03718309110892759


In [27]:
aapl_high = high['AAPL'].tail(60)
print('AAPL qualifying dates:', aapl_high[aapl_high].index)

print('AAPL Close x Volume, 31 December:', close['AAPL'].iloc[-1] * volume['AAPL'].iloc[-1])

AAPL qualifying dates: DatetimeIndex(['2024-11-25', '2024-12-20'], dtype='datetime64[ns]', name='Date', freq=None)
AAPL Close x Volume, 31 December: 9886756821.70868


In [29]:
tbl_data = features.copy()

tbl_data['Monetary'] = tbl_data['Monetary'] / 1e9
tbl_data['Mean_return'] = tbl_data['Mean_return'] * 100

tbl = tbl_data.describe()
tbl.loc['skew'] = tbl_data.skew()

display(tbl.round(3))

,Recency,Frequency,Monetary,Beta,Mean_return
count,486.000,486.000,486.000,486.000,486.000
mean,20.407,2.990,39.404,0.974,0.286
std,21.342,1.331,129.139,0.407,0.265
min,4.000,0.000,1.297,0.055,-0.447
25%,11.000,2.000,10.195,0.667,0.118
50%,11.000,3.000,16.684,0.951,0.262
75%,14.750,4.000,32.277,1.180,0.421
max,222.000,8.000,1847.697,3.779,1.777
skew,3.646,0.425,11.684,1.104,0.868




---

part 7